# BehaviorSense: Kaggle GPU backend (ONLINE / T4 ×2) for the competition demo

Run this notebook to obtain the Cloudflare backend link for `web/local_backend.py`.
Select **GPU T4 x2** and **Internet On**, and attach these public datasets:

1. [penguin069/behaviorsense-rgb-hybrid-serving](https://www.kaggle.com/datasets/penguin069/behaviorsense-rgb-hybrid-serving): runtime source, three trained heads, frozen thresholds, Swin3D-T and OSNet.
2. [krishnagupta02468/behavioursense-ww](https://www.kaggle.com/datasets/krishnagupta02468/behavioursense-ww): RTMO and RT-DETR detector weights.

Run the cells in order. Both T4s receive a model-forward smoke check before the API starts.
Keep the last cell running, copy its printed command, and execute it on your local machine.
Then serve the repository's `web/` folder and connect the page to `http://127.0.0.1:8899`.
The tunnel address changes with each session.

The notebook runs GPU stages 1-2. Your local bridge runs stages 3-4, report translation
and evidence verification using your local OpenRouter key. No Qwen model dataset is required.

The model preserves the saved 37.5% context / 37.5% temporal / 25% RGB blend, frozen
class thresholds and TV-plus-drinking rule. It performs offline multilabel inference.
T4 uses FP32 because it has no native BF16 support. The original BF16 test benchmark
(69.21% mAP, 28.12% pair F1) is a historical result with an unmet pair quality gate,
and does not measure this FP32 tracked-person adapter.

Default upload limits are 100 MB and 30 minutes at 20 FPS. Sources below 20 FPS
are refused. Only complete, sufficiently visible pose windows are classified.
No model training, threshold fitting or benchmark scoring happens in this notebook.


In [ ]:
# Locate and verify the runtime packaged with the model. Standard library only.
import hashlib, json, os, pathlib, sys
INPUT = pathlib.Path("/kaggle/input")
BUNDLE_DIR = None  # Optional explicit path if multiple serving bundles are attached.
mounts = sorted((INPUT / "datasets").glob("*/*")) + sorted(p for p in INPUT.iterdir() if p.is_dir())
if BUNDLE_DIR is None:
    candidates = sorted({base / prefix for base in mounts
        for prefix in ("", "bundle", "behaviorsense-rgb-hybrid-serving-20261016")
        if (base / prefix / "hybrid_manifest.json").is_file()})
    assert len(candidates) == 1, (
        "Attach the public serving bundle (hybrid_manifest.json must be at its root), "
        "or set BUNDLE_DIR explicitly. Found: " + str(candidates))
    BUNDLE_DIR = candidates[0]
BUNDLE_DIR = pathlib.Path(BUNDLE_DIR).resolve()
source_manifest = json.loads((BUNDLE_DIR / "source_manifest.json").read_text())
SOURCE = BUNDLE_DIR / "runtime_src"
for relative, expected in source_manifest["files"].items():
    path = (SOURCE / relative).resolve()
    assert path.is_relative_to(SOURCE.resolve()), "Source path escapes the bundle"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, "Source checksum differs: " + relative
sys.path.insert(0, str(SOURCE))
print("Serving bundle:", BUNDLE_DIR)
print("Verified runtime files:", len(source_manifest["files"]))


In [ ]:
# Install the CUDA 12.8 Torch runtime and CUDA 12 ONNX Runtime before importing models.
import importlib.metadata, subprocess
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return ""
if not (version("torch").startswith("2.11.") and "+cu128" in version("torch")
        and version("torchvision").startswith("0.26.") and "+cu128" in version("torchvision")):
    assert "torch" not in sys.modules, "Restart the Kaggle session before replacing its Torch runtime."
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "torch==2.11.0", "torchvision==0.26.0", "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "rtmlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "fastapi", "uvicorn", "nest_asyncio", "python-multipart", "pydantic>=2",
    "opencv-python-headless>=4.13,<4.15", "scipy", "PyYAML", "tqdm", "flatbuffers", "protobuf", "packaging"], check=True)
# Reinstall last: CPU and GPU ORT distributions share the same Python package directory.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
    "onnxruntime-gpu==1.26.0"], check=True)
import torch, torchvision, onnxruntime as ort
assert torch.cuda.device_count() >= 2, "Select GPU T4 x2 in Kaggle Settings, then start a fresh session."
assert "CUDAExecutionProvider" in ort.get_available_providers(), "ONNX Runtime CUDA is unavailable."
print("Torch", torch.__version__, "torchvision", torchvision.__version__, "ORT", ort.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(2)])


In [ ]:
# Resolve detector assets. No old ADL runs or language-model dataset is read.
from behaviorsense.models.rgb_hybrid import validate_bundle
from behaviorsense.models.visual_context import VALIDATED_MODEL_SHA256, file_sha256
from behaviorsense.service.rgb_hybrid_backend import ServingConfig, create_app, isolated_events
from behaviorsense.service.clip_workers import ClipWorker

def asset(filename, expected=None):
    candidates = sorted({base / prefix / filename for base in mounts
        for prefix in ("", "weights", "EmotionSense-Extended/weights")
        if (base / prefix / filename).is_file()})
    if expected:
        candidates = [path for path in candidates if file_sha256(path) == expected]
    assert candidates, "Attach behavioursense-ww containing the validated " + filename
    return candidates[0]

manifest = validate_bundle(BUNDLE_DIR)
config = ServingConfig(bundle=BUNDLE_DIR, rtmo=asset("rtmo-l.onnx"),
    rtdetr=asset("rtdetr-l.onnx", VALIDATED_MODEL_SHA256),
    max_frames=int(os.environ.get("BS_MAX_FRAMES", "36000")),
    max_upload_mb=int(os.environ.get("BS_MAX_UPLOAD_MB", "100")), token=os.environ.get("BS_TOKEN", ""))
print("Model:", manifest["model_id"])
print("Class decisions: frozen sigmoid thresholds; RGB encoder: FP32")
print("RTMO:", config.rtmo)
print("RT-DETR:", config.rtdetr)


In [ ]:
# Exercise RTMO, RT-DETR, Swin and all three heads on BOTH T4s before starting the service.
# Synthetic pixels only: no benchmark arrays, fitting or threshold changes.
import tempfile
smoke_receipts = []
for gpu in range(2):
    worker = ClipWorker(f"cuda:{gpu}", f"cuda:{gpu}", None)
    with tempfile.TemporaryDirectory(prefix="rgb-smoke-") as directory:
        for raw_event in isolated_events(config, worker, None, {}, directory):
            event = json.loads(raw_event)
            if event["event"] == "heartbeat":
                print(f"GPU {gpu}:", event.get("step"))
            if event["event"] == "result":
                receipt = event["payload"]
                assert receipt.get("smoke_passed"), receipt
                smoke_receipts.append(receipt)
                print(f"GPU {gpu} smoke passed:", receipt["gpu"], receipt["rgb_precision"])
assert len(smoke_receipts) == 2
app = create_app(config)
print("Ready for two simultaneous clips. Reports run through your local backend.")


In [ ]:
# Start the API and print the address to pass to your LOCAL backend.
import re, threading, time, urllib.request, uvicorn
CF = pathlib.Path("/kaggle/working/cloudflared")
if not CF.is_file():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/download/2024.12.2/cloudflared-linux-amd64", CF)
    CF.chmod(0o755)
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
for _ in range(100):
    if server.started:
        break
    if not server_thread.is_alive():
        raise RuntimeError("The API server exited during startup.")
    time.sleep(.1)
assert server.started, "The API server did not start."
log_path = pathlib.Path("/kaggle/working/rgb-cloudflared.log")
with log_path.open("w") as log:
    tunnel = subprocess.Popen([str(CF), "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
        stdout=log, stderr=subprocess.STDOUT)
    try:
        URL = None
        for _ in range(120):
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_path.read_text())
            if match:
                URL = match.group(0)
                break
            assert tunnel.poll() is None, "Cloudflare exited; inspect rgb-cloudflared.log."
            time.sleep(.5)
        assert URL, "No tunnel address appeared; inspect rgb-cloudflared.log."
        print("Run this command on your Windows machine:")
        print("python web/local_backend.py --kaggle " + URL + (" --token YOUR_BS_TOKEN" if config.token else ""))
        print("Connect the web page to http://127.0.0.1:8899 and leave this cell running.")
        print("No language-model dataset or API key is used by this notebook.")
        while tunnel.poll() is None and server_thread.is_alive():
            time.sleep(10)
    finally:
        server.should_exit = True
        if tunnel.poll() is None:
            tunnel.terminate()
            try:
                tunnel.wait(timeout=10)
            except subprocess.TimeoutExpired:
                tunnel.kill()
                tunnel.wait()
